# Python 入门实验

先在右上角选择 `Python (llm-learning)`，然后使用 **Run All**。每一节都修改参数再运行一次。

In [1]:
import sys
from pathlib import Path

print('Python 版本:', sys.version)
print('解释器路径:', sys.executable)
print('工作目录:', Path.cwd())
assert 'llm-learning' in sys.executable.lower(), '当前 Kernel 可能不是 llm-learning，请在右上角重新选择'

Python 版本: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:36:12) [MSC v.1944 64 bit (AMD64)]
解释器路径: d:\develop\miniforge3\envs\llm-learning\python.exe
工作目录: d:\workspace\llm-learning\01-Python工程基础


## 1. 变量、列表和字典

观察 Python 如何保存学习主题和配置。

In [2]:
topics = ['Python', 'RAG', 'LangGraph', 'MCP','hello','world']
config = {'week': 1, 'hours': 30, 'finished': False}

for index, topic in enumerate(topics, start=1):
    if len(topic) >3:
        print(f'{index}. {topic}')

print('每周小时:', config['hours'])

1. Python
3. LangGraph
5. hello
6. world
每周小时: 30


### 练习

增加两个主题；把 `hours` 改成你的实际时间；只打印名称长度大于 3 的主题。

In [3]:
def estimate_days(total_hours: float, hours_per_day: float = 4) -> float:
    if total_hours < 0:
        raise ValueError('total_hours 不能为负数')
    if hours_per_day <= 0:
        raise ValueError('hours_per_day 必须大于 0')
    return total_hours / hours_per_day

print('预计天数:', estimate_days(25))

预计天数: 6.25


## 2. JSON 读写

下面只在当前目录创建一个练习文件 `study_config.json`。

In [4]:
import json

output_path = Path('study_config.json')
output_path.write_text(
    json.dumps(config, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
loaded = json.loads(output_path.read_text(encoding='utf-8'))
print(loaded)

{'week': 1, 'hours': 30, 'finished': False}


## 3. Pydantic 运行时校验

如果 import 失败，请确认 Kernel；安装命令见环境安装教程。

In [5]:
from pydantic import BaseModel, Field, ValidationError

class StudyTask(BaseModel):
    title: str = Field(min_length=1, max_length=100)
    priority: int = Field(ge=0, le=3)
    hours: float = Field(gt=0, le=100)

task = StudyTask(title='学习 Python', priority=0, hours=8)
print(task.model_dump())

try:
    StudyTask(title='', priority=9, hours=-1)
except ValidationError as exc:
    print('预期中的校验错误:')
    print(exc)

{'title': '学习 Python', 'priority': 0, 'hours': 8.0}
预期中的校验错误:
3 validation errors for StudyTask
title
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short
priority
  Input should be less than or equal to 3 [type=less_than_equal, input_value=9, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal
hours
  Input should be greater than 0 [type=greater_than, input_value=-1, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than


## 4. 异步 I/O 直觉

两个各等待 1 秒的任务并发执行，整体通常略多于 1 秒，而不是 2 秒。

In [6]:
import asyncio
import time

async def fake_llm_call(name: str) -> str:
    await asyncio.sleep(1)
    return f'{name} 完成'

started = time.perf_counter()
results = await asyncio.gather(fake_llm_call('A'), fake_llm_call('B'))
print(results)
print('耗时:', round(time.perf_counter() - started, 2), '秒')

['A 完成', 'B 完成']
耗时: 1.01 秒


## 最终验收

1. 点击 Restart Kernel。
2. 点击 Run All。
3. 所有 Cell 从上到下无错误完成。
4. 删除练习产生的 `study_config.json`（如果不再需要）。
5. 在学习日志写下今天遇到的一个错误及定位过程。